# Strands Agent with Datadog Observability on Amazon Bedrock AgentCore Runtime

## Overview

This notebook deploys a [Strands](https://strandsagents.com/) agent to **Amazon Bedrock AgentCore Runtime** using the **AgentCore CLI**, with traces exported to **Datadog** via ADOT (AWS Distro for OpenTelemetry) auto-instrumentation — no Datadog Agent sidecar required.

## Key Components

- **Strands Agents** — Python framework for building LLM-powered agents with built-in telemetry
- **Amazon Bedrock AgentCore Runtime** — Managed runtime for hosting and scaling agents on AWS
- **AgentCore CLI** — CLI tool to create, develop, and deploy agents to AgentCore
- **ADOT** — AWS Distro for OpenTelemetry, auto-instruments the agent and exports traces via OTLP
- **Datadog** — Full-stack observability platform with APM, tracing, and AI-focused monitoring

## Prerequisites

- **Node.js** 20.x+ and **AgentCore CLI** installed (`npm install -g @aws/agentcore`)
- **uv** for Python dependency management ([install](https://docs.astral.sh/uv/getting-started/installation/))
- **AWS credentials** configured with Bedrock and AgentCore permissions
- **Datadog account** with an API key ([free trial](https://www.datadoghq.com/free-datadog-trial/))
- Access to **Amazon Bedrock Claude models** in your region

## Step 1: Configuration

Set your project name, Datadog credentials, and detect AWS account/region from your environment.

In [ ]:
import subprocess, json, os, boto3

PROJECT_NAME = "DatadogDemo"
WORK_DIR     = os.getcwd()
DD_SITE      = "datadoghq.com"
DD_API_KEY   = os.getenv("DD_API_KEY", "")  # Replace or set DD_API_KEY env var
AGENTCORE    = "/usr/local/bin/agentcore"

sts = boto3.client("sts")
AWS_ACCOUNT = sts.get_caller_identity()["Account"]
AWS_REGION = boto3.session.Session().region_name or "us-east-1"

def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return r

project_dir = os.path.join(WORK_DIR, PROJECT_NAME)
agent_dir = os.path.join(project_dir, "app", PROJECT_NAME)

print(f"Project: {PROJECT_NAME}")
print(f"AWS:     {AWS_ACCOUNT} / {AWS_REGION}")
print(f"Datadog: {DD_SITE}")

## Step 2: Create AgentCore Project

Scaffold a new AgentCore project using the CLI. This generates:
- A **Strands** agent template with Amazon Bedrock as the model provider
- A **Container** build configuration (Dockerfile, pyproject.toml)
- CDK infrastructure code for deployment

In [ ]:
run(
    f"{AGENTCORE} create --name {PROJECT_NAME} --framework Strands "
    f"--model-provider Bedrock --build Container --memory none "
    f"--skip-git --skip-python-setup --json",
    cwd=WORK_DIR
)

## Step 3: Configure AWS Deployment Target

Write the `aws-targets.json` file with your AWS account ID and region. This tells the CLI where to deploy the CloudFormation stack.

In [ ]:
targets = [{"name": "default", "account": AWS_ACCOUNT, "region": AWS_REGION}]
with open(os.path.join(project_dir, "agentcore", "aws-targets.json"), "w") as f:
    json.dump(targets, f, indent=2)
print(f"Target: {AWS_ACCOUNT} / {AWS_REGION}")

## Step 4: Configure Datadog OTEL Export

Inject OpenTelemetry environment variables into the agent configuration. At runtime, ADOT reads these variables and auto-instruments all Bedrock model calls, tool invocations, and agent lifecycle spans — exporting them directly to Datadog's OTLP intake.

| Variable | Purpose |
|---|---|
| `AGENT_OBSERVABILITY_ENABLED` | Enables ADOT auto-instrumentation in AgentCore Runtime |
| `OTEL_EXPORTER_OTLP_TRACES_ENDPOINT` | Datadog's OTLP intake URL |
| `OTEL_EXPORTER_OTLP_TRACES_HEADERS` | API key and source tag for Datadog |
| `OTEL_PYTHON_DISTRO` / `OTEL_PYTHON_CONFIGURATOR` | Selects the AWS ADOT distro |
| `OTEL_SERVICE_NAME` | Service name shown in Datadog APM |

In [ ]:
spec_path = os.path.join(project_dir, "agentcore", "agentcore.json")
with open(spec_path) as f:
    spec = json.load(f)

for agent in spec["agents"]:
    agent["envVars"] = [
        {"name": "AGENT_OBSERVABILITY_ENABLED", "value": "true"},
        {"name": "OTEL_EXPORTER_OTLP_PROTOCOL", "value": "http/protobuf"},
        {"name": "OTEL_EXPORTER_OTLP_TRACES_ENDPOINT", "value": f"https://otlp.{DD_SITE}/v1/traces"},
        {"name": "OTEL_EXPORTER_OTLP_TRACES_HEADERS", "value": f"dd-api-key={DD_API_KEY},dd-otlp-source=llmobs"},
        {"name": "OTEL_EXPORTER_OTLP_TRACES_PROTOCOL", "value": "http/protobuf"},
        {"name": "OTEL_PYTHON_CONFIGURATOR", "value": "aws_configurator"},
        {"name": "OTEL_PYTHON_DISTRO", "value": "aws_distro"},
        {"name": "OTEL_SERVICE_NAME", "value": "agentcore-datadog-demo"},
    ]

with open(spec_path, "w") as f:
    json.dump(spec, f, indent=2)
print("Updated agentcore.json with Datadog OTEL env vars")

## Step 5: Write Agent Code

Replace the template agent with a custom Strands agent that includes:
- A **calculator** tool (built-in from `strands_tools`)
- A custom **weather** tool
- Claude Sonnet as the model via Amazon Bedrock

The `@app.entrypoint` decorator is the AgentCore Runtime contract — it receives invocation payloads and returns the agent's response.

In [ ]:
with open(os.path.join(agent_dir, "main.py"), "w") as f:
    f.write('''\
"""Strands agent with Datadog observability on Amazon Bedrock AgentCore Runtime."""

import os
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)


@tool
def weather() -> str:
    """Get the current weather."""
    return "sunny"


model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-20250514-v1:0")
model = BedrockModel(model_id=model_id)

agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You are a helpful assistant. You can do simple math calculations and tell the weather.",
)

app = BedrockAgentCoreApp()


@app.entrypoint
def handler(payload: dict, context=None) -> str:
    """Handle incoming agent invocations."""
    user_input = payload.get("prompt", "Hello!")
    logger.info("Processing prompt: %s", user_input[:100])
    response = agent(user_input)
    return response.message["content"][0]["text"]


if __name__ == "__main__":
    app.run()
''')
print("Wrote main.py")

## Step 6: Add Dependencies

Add `strands-agents-tools` (provides the `calculator` tool) to the project's `pyproject.toml`, then generate the `uv.lock` file required by the Container build's Dockerfile.

In [ ]:
pyproject_path = os.path.join(agent_dir, "pyproject.toml")
with open(pyproject_path) as f:
    content = f.read()

if "strands-agents-tools" not in content:
    content = content.replace(
        '"strands-agents >= 1.13.0",',
        '"strands-agents >= 1.13.0",\n    "strands-agents-tools",'
    )
    with open(pyproject_path, "w") as f:
        f.write(content)
    print("Added strands-agents-tools")

In [ ]:
run("uv lock", cwd=agent_dir)

## Step 7: Deploy to AgentCore Runtime

Deploy the agent to AWS. The CLI will:
1. Package the agent code into a container image via **CodeBuild**
2. Push the image to **ECR**
3. Deploy the **CloudFormation** stack with the AgentCore Runtime, IAM roles, and observability configuration

This typically takes 5–10 minutes on the first deploy.

In [ ]:
run(f"{AGENTCORE} deploy -y --json", cwd=project_dir)

## Step 8: Invoke the Agent

Send prompts to the deployed agent. Each invocation generates traces that are exported to both **CloudWatch** (automatic from AgentCore) and **Datadog** (via ADOT OTLP export).

In [ ]:
run(f'{AGENTCORE} invoke "What is 42 * 17?" --json', cwd=project_dir)

In [ ]:
run(f'{AGENTCORE} invoke "What is the weather now?" --json', cwd=project_dir)

## Step 9: View Traces in Datadog

Open your [Datadog APM dashboard](https://app.datadoghq.com/apm/traces) and filter by `service:agentcore-datadog-demo`.

You should see traces for each agent invocation with spans for:
- Agent invocation lifecycle
- Model calls (Bedrock `InvokeModel`)
- Tool execution (calculator, weather)
- Token usage and latency metrics

<!-- TODO: Replace with actual screenshot -->
![Datadog APM Dashboard](images/datadog_dashboard.png)
*Datadog APM showing AgentCore agent traces with LLM and tool spans.*

## Cleanup

Tear down the CloudFormation stack and remove the local project directory.

In [ ]:
import shutil
try:
    run(f"{AGENTCORE} remove all -y --json", cwd=project_dir)
    run(f"{AGENTCORE} deploy -y --json", cwd=project_dir)
except RuntimeError:
    print("Teardown via CLI failed, deleting stack directly...")
    run(f"aws cloudformation delete-stack --stack-name AgentCore-{PROJECT_NAME}-default --region {AWS_REGION}")
    run(f"aws cloudformation wait stack-delete-complete --stack-name AgentCore-{PROJECT_NAME}-default --region {AWS_REGION}")
shutil.rmtree(project_dir, ignore_errors=True)
print(f"Cleaned up {project_dir}")

## Summary

This notebook demonstrated:

- Deploying a **Strands agent** to **Amazon Bedrock AgentCore Runtime** using the **AgentCore CLI**
- **Container build** for full control over the runtime environment and reproducible deployments
- **ADOT auto-instrumentation** routing traces to **Datadog** via OTLP — no sidecar agent needed
- **Dual observability**: CloudWatch (automatic from AgentCore) + Datadog (via ADOT OTLP export)
- **Infrastructure-as-code** deployment via CDK/CloudFormation with drift detection and rollback